In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_network import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
  -> ny bästa modell sparad till ../models/convolution_model.pth
Epoch   0 | train: 1.0837 | val: 0.6543 | acc: 66.14% | AUC: 0.720  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model.pth
Epoch   1 | train: 0.5909 | val: 0.4599 | acc: 92.94% | AUC: 0.848  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model.pth
Epoch   2 | train: 0.4741 | val: 0.4104 | acc: 87.04% | AUC: 0.888  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model.pth
Epoch   3 | train: 0.4430 | val: 0.3092 | acc: 96.38% | AUC: 0.889  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model.pth
Epoch   4 | train: 0.4274 | val: 0.2940 | acc: 96.01% | AUC: 0.894  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model.pth
Epoch   5 | train: 0.4183 | val: 0.3595 | acc: 90.42% | AUC: 0.903  | LR: 0.001
  -> ny bästa modell sparad till ../mode

In [2]:
con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   0 | train: 1.0603 | val: 0.6655 | acc: 87.37% | AUC: 0.601  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   1 | train: 0.5972 | val: 0.5473 | acc: 86.30% | AUC: 0.786  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   2 | train: 0.4742 | val: 0.4610 | acc: 89.18% | AUC: 0.839  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   3 | train: 0.4363 | val: 0.4367 | acc: 87.27% | AUC: 0.855  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   4 | train: 0.4197 | val: 0.4338 | acc: 90.76% | AUC: 0.860  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_fold1.pth
Epoch   5 | train

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.100574,0.147737,95.308871,0.987837,0.953266,0.951852,1795,88,13,257
1,2,0.107849,0.130632,92.893637,0.992697,0.921402,0.981481,1735,148,5,265
2,3,0.106001,0.211678,97.214485,0.983053,0.987798,0.862454,1862,23,37,232
3,4,0.086773,0.196693,95.724907,0.978037,0.963356,0.914498,1814,69,23,246
4,5,0.099052,0.151222,95.030190,0.988524,0.950106,0.951673,1790,94,13,256
5,6,0.090739,0.118243,95.961003,0.990337,0.963395,0.933086,1816,69,18,251
6,7,0.103581,0.150353,94.144981,0.982068,0.943176,0.929368,1776,107,19,250
7,8,0.102991,0.206852,96.564531,0.979870,0.980902,0.858736,1849,36,38,231
8,9,0.177939,0.236998,88.388295,0.976042,0.873673,0.955390,1646,238,12,257
9,10,0.113186,0.135546,93.779016,0.989051,0.932626,0.973978,1758,127,7,262


In [3]:
test_rows = test_rows[test_rows['label'].isin([0,1])]
result = CNN.predict(test_rows)
_ = evaluate(result)

KeyError: 'prediction'